#  Week 5, Day 6 — June 20, 2026


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns, os, json, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {"raw":BASE_DIR+"/data/raw","processed":BASE_DIR+"/data/processed",
        "metadata":BASE_DIR+"/data/metadata","visualizations":BASE_DIR+"/visualizations"}
print("Environment ready ")

In [ ]:
import datetime
cluster_df = pd.read_csv(DIRS['processed']+'/cluster_results.csv')
with open(DIRS['metadata']+'/kmeans_model_meta.json')  as f: km_meta  = json.load(f)
with open(DIRS['metadata']+'/regression_results.json') as f: reg_meta = json.load(f)
with open(DIRS['metadata']+'/correlation_results.json')as f: cor_meta = json.load(f)
print('All Week 5 metadata loaded ')

## Step 1: Centroid Table — Original Scale

In [ ]:
FEATURES = ['SST (°C)', 'pH Level', 'Species Observed']
profiles = cluster_df.groupby('Risk_Cluster')[FEATURES].mean()
counts   = cluster_df['Risk_Cluster'].value_counts()
total    = len(cluster_df)

print('CLUSTER CENTROID TABLE — ORIGINAL UNITS')
print('='*75)
print(f'  {"Risk Zone":<12} {"SST (°C)":>10} {"pH Level":>10} {"Species":>12}  n     %   SST margin  pH margin')
print('-'*75)

TIP_SST = 30.6286; TIP_pH = 7.879525
cluster_summary = []
for risk in ['Stable','At_Risk','Critical']:
    if risk not in profiles.index: continue
    row = profiles.loc[risk]; n = counts[risk]; pct = n/total*100
    sst_margin = TIP_SST - row['SST (°C)']
    ph_margin  = row['pH Level'] - TIP_pH
    print(f'  {risk:<12} {row["SST (°C)"]:>10.3f} {row["pH Level"]:>10.5f} '
          f'{row["Species Observed"]:>12.2f}  {n}  ({pct:.1f}%)  {sst_margin:+.3f}°C   {ph_margin:+.5f}')
    cluster_summary.append({'Risk_Zone':risk,'n':int(n),'Pct':round(pct,1),
        'Centroid_SST_C':round(row['SST (°C)'],3),'Centroid_pH':round(row['pH Level'],5),
        'Centroid_Species':round(row['Species Observed'],2),
        'SST_Tipping_Margin':round(sst_margin,3),'pH_Tipping_Margin':round(ph_margin,5)})
print()
print('INTERPRETATION:')
print('  Stable   (27.05°C, pH 8.097): 3.58°C from SST tipping — safe zone')
print('  At-Risk  (28.65°C, pH 8.043): 1.98°C from SST tipping — monitoring needed')
print('  Critical (30.42°C, pH 7.997): only 0.20°C from SST tipping — immediate concern')

## Step 2: Risk Zone Summary Export for Dashboard

In [ ]:
risk_summary = pd.DataFrame(cluster_summary)
risk_summary.to_csv(DIRS['processed']+'/risk_zone_summary.csv', index=False)
print('risk_zone_summary.csv saved   (Power BI / Tableau ready)')
print()
print(risk_summary.to_string(index=False))

## Step 3: Comprehensive Modelling Report

In [ ]:
report_path = DIRS['metadata']+'/statistical_modelling_report_week5.txt'
with open(report_path,'w') as f:
    f.write('STATISTICAL MODELLING REPORT — WEEK 5')
    f.write('HCLTech Internship | Aditya Saxena | A010145025085')
    f.write(f'Generated: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}')

    f.write('SECTION 1: PEARSON CORRELATION ANALYSIS')
    f.write('-'*50+'')
    f.write('Dataset: realistic_ocean_climate_dataset (n=500)')
    for res in cor_meta:
        f.write(f'  {res["pair"]}:')
        f.write(f'    r={res["r"]}  p={res["p"]:.2e}  n={res["n"]}')
        f.write(f'    {res["strength"]} {res["direction"]} correlation')
    f.write('  KEY FINDING: SST is the dominant stressor.')
    f.write('  r=-0.6813: each +1°C SST rise = ~9.79 fewer species observed.')
    f.write('  SST also reduces pH (r=-0.5154) — compounding dual stressor effect.')

    f.write('SECTION 2: LINEAR REGRESSION — ECOLOGICAL TIPPING POINTS')
    f.write('-'*50+'')
    m1 = reg_meta['model_SST_Species']; m2 = reg_meta['model_pH_Species']
    f.write(f'  Model 1 (SST -> Species):
    {m1["equation"]}')
    f.write(f'    R2={m1["R2"]}  MAE={m1["MAE"]} species')
    f.write(f'    Tipping Point: SST={m1["tipping_point_SST_C"]}°C  Margin: {m1["margin_C"]}°C ')


    f.write(f'  Model 2 (pH -> Species):
    {m2["equation"]}')
    f.write(f'    R2={m2["R2"]}  MAE={m2["MAE"]} species')
    f.write(f'    Tipping Point: pH={m2["tipping_point_pH"]}  Margin: {m2["margin_pH"]} units')

    f.write('SECTION 3: K-MEANS RISK CLUSTERING')
    f.write('-'*50+'')
    f.write(f'  Model: KMeans(k=3, random_state=42, n_init=10)')
    f.write(f'  Final inertia: {km_meta["inertia"]}  Iterations: {km_meta["n_iter"]}')
    f.write(f'  Silhouette Score (overall): 0.2922 (Moderate)')
    f.write('  CENTROIDS (original scale):')
    for row in cluster_summary:
        f.write(f'    {row["Risk_Zone"]:<10}: SST={row["Centroid_SST_C"]}°C, '
                f'pH={row["Centroid_pH"]}, Species={row["Centroid_Species"]}, '
                f'n={row["n"]} ({row["Pct"]}%)')
    f.write('
  RISK DISTRIBUTION:')
    f.write('    Stable   : 147 records (29.4%) — safe zone, SST margin +3.58°C')
    f.write('    At-Risk  : 252 records (50.4%) — monitoring required, margin +1.98°C')
    f.write('    Critical : 101 records (20.2%) — immediate concern, margin +0.20°C')

    f.write('SECTION 4: INTEGRATED ECOLOGICAL RISK ASSESSMENT')
    f.write('-'*50+'')
    for insight in [
        'Arabian Sea (35.82% plastic share) overlaps with At-Risk SST zone — priority region.',
        '20.2% of observations already in Critical SST range (centroid 30.42°C).',
        'SST tipping point (30.63°C) only 2.13°C above current mean — ~10 year horizon.',
        'pH tipping point is ~85 years away — SST is the near-term priority stressor.',
        'Critical zone centroid (30.42°C) is only 0.20°C below tipping point.',
    ]:
        f.write(f'  - {insight}')

print(f'Modelling report saved ')
with open(report_path) as f: print(f.read())